# 🏉 Bradford Bulls - AI Logo Exposure Detection Pipeline

Đây là notebook hoàn chỉnh triển khai toàn bộ pipeline đo lường ROI của Sponsor Logo trên kit thi đấu của Bradford Bulls.

Notebook này được thiết kế để chạy trên **Google Colab (GPU)**.


## Bước 1: Setup Môi Trường & Mount Google Drive
Kết nối với Google Drive để lấy video input và lưu trữ kết quả output.


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Định nghĩa các đường dẫn thư mục trên Drive
# VUI LÒNG ĐỔI TÊN THƯ MỤC NÀY THÀNH THƯ MỤC CHỨA DATA CỦA BẠN TRÊN DRIVE
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Bradford_Bulls_AI'
VIDEO_DIR = os.path.join(DRIVE_PROJECT_DIR, 'videos')
OUTPUT_DIR = os.path.join(DRIVE_PROJECT_DIR, 'outputs')

# Tạo thư mục output nếu chưa có
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Data directory: {DRIVE_PROJECT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


## Bước 2: Cài Đặt Thư Viện
Cài đặt YOLOv8/v11 (Ultralytics), ByteTrack, OpenCV và các thư viện cần thiết.


In [ ]:
!pip install ultralytics opencv-python-headless scenedetect filterpy
!pip install lap # For ByteTrack assignment


## Bước 3: Import Libraries & Define Configuration
Import các module và cấu hình thông số cho các Sponsor Logos.


In [ ]:
import cv2
import json
import numpy as np
import pandas as pd
from datetime import datetime
from collections import defaultdict
from ultralytics import YOLO
from scenedetect import detect, ContentDetector
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
SPONSOR_CONFIG = {
    "topnotch": {"name": "Top Notch", "kit": "home", "pos": "chest_front", "weight": 0.26},
    "floor_tonic": {"name": "Floor Tonic", "kit": "away", "pos": "chest_front", "weight": 0.26},
    "mcp": {"name": "MCP", "kit": "both", "pos": "back_upper", "weight": 0.08},
    "romantica": {"name": "Romantica Beds", "kit": "both", "pos": "chest_left", "weight": 0.07},
    "bartercard": {"name": "Bartercard", "kit": "both", "pos": ["chest_right", "back_mid"], "weight": 0.11},
    "acs_group": {"name": "ACS Group", "kit": "both", "pos": "back_lower", "weight": 0.03},
    "klg_europe": {"name": "KLG Europe", "kit": "both", "pos": "shorts_back_left", "weight": 0.05},
    "aon": {"name": "AON", "kit": "both", "pos": "shorts_back_right", "weight": 0.03},
    # Thêm các nhà tài trợ khác từ CSV vào đây...
}

# Thresholds
CONF_PLAYER = 0.45
CONF_POSE = 0.30
BLUR_THRESHOLD = 80.0
SAMPLE_RATE = 2 # Process 2 frames per second to save time and compute


## Bước 4: Định nghĩa Core Classes cho Pipeline
Bao gồm Player Tracker, Pose ROI Extractor, và Quality Scorer.


In [ ]:
class SceneManager:
    """Lọc các đoạn live play, bỏ qua replay và commercial."""
    def __init__(self, video_path):
        self.video_path = video_path
        
    def get_live_play_segments(self):
        # Trả về toàn bộ video, vì logo trên áo đội target trong Replay VẪN CÓ GIÁ TRỊ.
        # Ở version nâng cao, module này có thể lọc bỏ thời gian nghỉ giữa hiệp.
        print(f"Analyzing scenes for {os.path.basename(self.video_path)}...")
        cap = cv2.VideoCapture(self.video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        return [(0, total_frames)]

class PoseROIExtractor:
    """Map 17 keypoints của YOLO-Pose thành các vùng (regions) trên jersey."""
    def __init__(self):
        # Keypoint indices for YOLO Pose
        self.NOSE = 0
        self.L_SHOULDER, self.R_SHOULDER = 5, 6
        self.L_ELBOW, self.R_ELBOW = 7, 8
        self.L_HIP, self.R_HIP = 11, 12
        
    def extract_regions(self, keypoints, bbox):
        x1, y1, x2, y2 = bbox
        kpts = keypoints # shape (17, 3) -> x, y, conf
        
        regions = {}
        # Ngưỡng tin cậy của keypoint
        if kpts[self.L_SHOULDER][2] > 0.4 and kpts[self.R_SHOULDER][2] > 0.4 and \
           kpts[self.L_HIP][2] > 0.4 and kpts[self.R_HIP][2] > 0.4:
            
            # Tính Chest Region (Front)
            chest_y1 = min(kpts[self.L_SHOULDER][1], kpts[self.R_SHOULDER][1])
            chest_y2 = max(kpts[self.L_HIP][1], kpts[self.R_HIP][1])
            chest_x1 = min(kpts[self.L_SHOULDER][0], kpts[self.L_HIP][0])
            chest_x2 = max(kpts[self.R_SHOULDER][0], kpts[self.R_HIP][0])
            
            regions['chest'] = [int(chest_x1), int(chest_y1), int(chest_x2), int(chest_y2)]
            
            # Tính Back Region
            facing_front = kpts[self.NOSE][2] > 0.5
            if not facing_front:
                regions['back'] = regions['chest']
                del regions['chest']
                
        return regions

class QualityScoringEngine:
    """Tính toán Quality Score cho mỗi frame logo xuất hiện."""
    def calculate_score(self, logo_bbox, frame_shape, image_crop):
        # 1. Size Score
        frame_area = frame_shape[0] * frame_shape[1]
        logo_area = (logo_bbox[2] - logo_bbox[0]) * (logo_bbox[3] - logo_bbox[1])
        size_ratio = logo_area / frame_area
        size_score = min(size_ratio * 50, 1.0) 
        
        # 2. Clarity Score
        gray = cv2.cvtColor(image_crop, cv2.COLOR_BGR2GRAY)
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        clarity_score = 1.0 if laplacian_var > BLUR_THRESHOLD else laplacian_var / BLUR_THRESHOLD
        
        # 3. Position Score
        center_x, center_y = frame_shape[1] / 2, frame_shape[0] / 2
        l_x = (logo_bbox[0] + logo_bbox[2]) / 2
        l_y = (logo_bbox[1] + logo_bbox[3]) / 2
        dist = np.sqrt((l_x - center_x)**2 + (l_y - center_y)**2)
        max_dist = np.sqrt(center_x**2 + center_y**2)
        position_score = max(0, 1.0 - (dist / max_dist))
        
        # Weighted sum
        final_score = (size_score * 0.3) + (clarity_score * 0.4) + (position_score * 0.3)
        return final_score



## Bước 5: Initialize Models
Tải mô hình YOLOv8 Pose (để tìm cầu thủ và khung xương) và mô hình nhận diện Logo.


In [ ]:
# Load YOLOv8 Pose (Tự động tải weights nếu chưa có)
pose_model = YOLO("yolov8n-pose.pt") # Khuyên dùng yolov8m-pose hoặc yolov8x-pose cho accuracy cao hơn

class LogoRecognizer:
    def __init__(self):
        pass
        
    def detect_logos(self, player_crop, regions):
        """
        Giả lập việc detect logos. Trong thực tế, bạn sẽ dùng YOLO-OBB hoặc Template Matching tại đây.
        """
        detected = []
        if 'chest' in regions:
            detected.append({
                'sponsor': 'topnotch', 
                'bbox': regions['chest'], 
                'conf': 0.90
            })
        if 'back' in regions:
            detected.append({
                'sponsor': 'mcp', 
                'bbox': regions['back'], 
                'conf': 0.85
            })
        return detected

logo_model = LogoRecognizer()



## Bước 6: Main Processing Pipeline
Hàm xử lý chính cho video trận đấu.


In [ ]:
def process_match_video(video_path, kit_type="home"):
    if not os.path.exists(video_path):
        print(f"Lỗi: Không tìm thấy file video {video_path}")
        return None, None
        
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps / SAMPLE_RATE)
    
    scene_mgr = SceneManager(video_path)
    live_segments = scene_mgr.get_live_play_segments()
    
    roi_extractor = PoseROIExtractor()
    scoring_engine = QualityScoringEngine()
    
    exposure_data = defaultdict(float)
    raw_duration = defaultdict(float)
    
    frame_count = 0
    processed_count = 0
    
    print(f"Starting processing: {os.path.basename(video_path)} at {SAMPLE_RATE} FPS")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        if frame_count % frame_interval != 0:
            continue
            
        processed_count += 1
        
        # 1. Player & Pose Detection
        results = pose_model(frame, verbose=False, conf=CONF_PLAYER)
        
        for r in results:
            boxes = r.boxes.xyxy.cpu().numpy()
            keypoints = r.keypoints.data.cpu().numpy() 
            
            for i, box in enumerate(boxes):
                x1, y1, x2, y2 = map(int, box)
                kpts = keypoints[i]
                
                player_crop = frame[y1:y2, x1:x2]
                if player_crop.size == 0: continue
                
                # 2. Extract Regions
                rel_kpts = kpts.copy()
                rel_kpts[:, 0] -= x1
                rel_kpts[:, 1] -= y1
                regions = roi_extractor.extract_regions(rel_kpts, [0, 0, x2-x1, y2-y1])
                
                # 3. Logo Recognition
                logos = logo_model.detect_logos(player_crop, regions)
                
                # 4. Calculate Quality Score
                for logo in logos:
                    sponsor = logo['sponsor']
                    
                    if kit_type == "away" and sponsor == "topnotch":
                        sponsor = "floor_tonic"
                    
                    lx1, ly1, lx2, ly2 = logo['bbox']
                    logo_crop = player_crop[ly1:ly2, lx1:lx2]
                    
                    if logo_crop.size > 0:
                        q_score = scoring_engine.calculate_score(logo['bbox'], frame.shape, logo_crop)
                        time_increment = 1.0 / SAMPLE_RATE
                        raw_duration[sponsor] += time_increment
                        exposure_data[sponsor] += (time_increment * q_score)
                        
        if processed_count % 50 == 0:
            print(f"Processed {processed_count} sampled frames...")
            
    cap.release()
    print("Processing complete!")
    return raw_duration, exposure_data



## Bước 7: Thực thi & Lưu báo cáo vào Google Drive
Chạy pipeline trên video và lưu cả File CSV (báo cáo data) lẫn biểu đồ Hình ảnh trực tiếp vào Drive của bạn.


In [ ]:
# CẤU HÌNH ĐẦU VÀO TỪ DRIVE
TARGET_VIDEO = "M06_black_1080p.mp4" 
KIT_TYPE = "away"

# Tạo đường dẫn tuyệt đối trong Drive
full_video_path = os.path.join(VIDEO_DIR, TARGET_VIDEO)

# --- THỰC THI CHÍNH ---
# Uncomment 2 dòng dưới để chạy thật nếu bạn đã có file video trong Drive:
# print(f"Chuẩn bị xử lý file: {full_video_path}")
# raw_durations, weighted_durations = process_match_video(full_video_path, KIT_TYPE)

# --- GIẢ LẬP KẾT QUẢ ĐỂ DEMO (Nếu chưa có video upload lên Drive) ---
raw_durations = {"floor_tonic": 450.5, "mcp": 320.0, "romantica": 210.0, "acs_group": 150.0}
weighted_durations = {"floor_tonic": 310.2, "mcp": 180.5, "romantica": 120.3, "acs_group": 95.0}
# -------------------------------------------------------------------

if raw_durations and weighted_durations:
    # TÍNH TOÁN ROI (MVE - Media Value Equivalent)
    COST_PER_30S_AD = 500.0 # Giá giả định £500 mỗi 30s
    
    results_list = []
    for sponsor in raw_durations.keys():
        raw_sec = raw_durations[sponsor]
        weighted_sec = weighted_durations[sponsor]
        mve = (weighted_sec / 30.0) * COST_PER_30S_AD
        
        results_list.append({
            "Sponsor": SPONSOR_CONFIG.get(sponsor, {}).get("name", sponsor),
            "Raw Exposure (s)": round(raw_sec, 1),
            "Weighted Quality Time (s)": round(weighted_sec, 1),
            "Avg Quality Score (%)": round((weighted_sec / raw_sec) * 100, 1) if raw_sec > 0 else 0,
            "Est. Media Value (£)": f"£{mve:,.2f}"
        })
    
    df_results = pd.DataFrame(results_list)
    df_results = df_results.sort_values(by="Weighted Quality Time (s)", ascending=False)
    
    print("\n" + "="*50)
    print("📊 BÁO CÁO PHÂN TÍCH LOGO EXPOSURE CHUNG CUỘC")
    print("="*50)
    display(df_results)
    
    # XUẤT FILE BÁO CÁO CSV VÀO DRIVE
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"{TARGET_VIDEO.split('.')[0]}_exposure_report_{timestamp}.csv"
    csv_path = os.path.join(OUTPUT_DIR, csv_filename)
    df_results.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n✅ Đã lưu báo cáo Data (CSV) tại: {csv_path}")
    
    # VẼ BIỂU ĐỒ VÀ LƯU HÌNH ẢNH VÀO DRIVE
    plt.figure(figsize=(10, 6))
    sponsors = df_results['Sponsor']
    raw_vals = df_results['Raw Exposure (s)']
    weighted_vals = df_results['Weighted Quality Time (s)']
    
    x = np.arange(len(sponsors))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(12, 6))
    rects1 = ax.bar(x - width/2, raw_vals, width, label='Raw Exposure', color='#cccccc')
    rects2 = ax.bar(x + width/2, weighted_vals, width, label='Weighted (Quality) Exposure', color='#e32219')
    
    ax.set_ylabel('Thời gian (Giây)')
    ax.set_title(f'Raw vs Weighted Exposure Time per Sponsor - {TARGET_VIDEO}')
    ax.set_xticks(x)
    ax.set_xticklabels(sponsors, rotation=45, ha='right')
    ax.legend()
    
    plt.tight_layout()
    
    # Save Image to Drive
    plot_filename = f"{TARGET_VIDEO.split('.')[0]}_exposure_chart_{timestamp}.png"
    plot_path = os.path.join(OUTPUT_DIR, plot_filename)
    fig.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✅ Đã lưu biểu đồ (PNG) tại: {plot_path}")
    
    plt.show()
else:
    print("Không có kết quả nào để hiển thị.")

